# Review binary alarm forecasts without retraining

This notebook reads only saved reports and predictions. It does not fit models, modify thresholds, or reopen source test data. All reported metrics concern eligible 10-minute **windows**, not independent alarm events.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'configs' / 'short_horizon_alarm.json').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from inside the Git repository.')
sys.path.insert(0, str(REPO / 'src'))
from wt_predmaint.short_horizon import load_config
CFG = load_config(REPO / 'configs' / 'short_horizon_alarm.json')
print('Repository:', REPO)


In [ ]:
REPORT_ROOT = REPO / CFG['report_root']
RUN_ID = None  # Paste a folder name to review a specific run.
if RUN_ID is None:
    completed = [p for p in REPORT_ROOT.iterdir() if (p / 'status.json').is_file()
                 and json.loads((p / 'status.json').read_text()).get('status') in ['complete', 'frozen_before_test']]
    if not completed:
        raise FileNotFoundError('No completed run yet. Run notebook 01 first.')
    OUT = sorted(completed)[-1]
else:
    OUT = REPORT_ROOT / RUN_ID
print('Reviewing:', OUT)


In [ ]:
display(pd.Series(json.loads((OUT / 'configuration.json').read_text()), name='configuration').to_frame())
display(pd.DataFrame(json.loads((OUT / 'sampling.json').read_text())).T)


## Data inspection and visualisation

In [ ]:
display(pd.read_csv(OUT / 'data_quality.csv'))
for filename in ['01_coverage_missingness.png', '02_alarm_prevalence.png', '03_sensors_and_alarm_starts.png']:
    display(Image(filename=str(OUT / filename)))


## Validation-selected models and final test performance

In [ ]:
selected = json.loads((OUT / 'frozen_selection.json').read_text())
display(pd.DataFrame(selected).T)
display(pd.read_csv(OUT / 'validation_metrics.csv'))
if (OUT / 'test_metrics.csv').is_file():
    test = pd.read_csv(OUT / 'test_metrics.csv')
    display(test)
    display(test.loc[[row.model == selected[str(row.horizon_minutes)]['model'] for row in test.itertuples()]])
else:
    print('This run has validation results only; test has not been evaluated.')
for split in ['validation', 'test']:
    if not (OUT / f'{split}_metrics.csv').is_file():
        continue
    for name in ['precision_recall', 'confusion_matrices']:
        number = '04' if name == 'precision_recall' else '05'
        display(Image(filename=str(OUT / f'{number}_{split}_{name}.png')))
if (OUT / '06_validation_forecast_timeline.png').is_file():
    display(Image(filename=str(OUT / '06_validation_forecast_timeline.png')))


## Which sensors did the selected models use?

These are training-derived model importance measures, not causal effects.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 6), constrained_layout=True)
for ax, horizon in zip(axes, [10, 20, 30], strict=False):
    name = selected[str(horizon)]['model']
    importance = pd.read_csv(OUT / f'{horizon}min_{name}_feature_importance.csv')
    importance.head(15).sort_values('importance').plot.barh(x='feature', y='importance', ax=ax, legend=False)
    ax.set_title(f'{horizon} min: {name}')
plt.show()
